In [2]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from recOrder.recOrder.io.reader import MicromanagerReader
from recOrder.recOrder.io.writer import WaveorderWriter

from pycromanager.data import Dataset

## Gather Data Paths

In [3]:
data_dir = '/gpfs/CompMicro/rawdata/hummingbird/Cameron/2021_05_18_MouseBrain_MBP_5x/Cloz_MBP_1'
# well_name = 'Well_0'

# time_folders = glob.glob(data_dir+'*'+well_name+'*')

bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Cameron/2021_05_18_MouseBrain_MBP_5x/BG/'
bg_data = load_bg(bg_path, 2048, 2048, ROI = (225, 342, 1419, 1446), cropped = True)

## Initialize Reconstruction

In [4]:
## Set Reconstruction Parameters
image_dim = (2048,2048)
wavelength = 525
swing = 0.1
N_channel = 4
NA_obj = 
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1.33


reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.59 min)


# Initialize Writer

In [ ]:
save_dir = '/gpfs/CompMicro/projects/A549/2021_04_06_A549_Live_63x_04NA'

writer_physical = WaveorderWriter(save_dir, 'physical')
writer_stokes = WaveorderWriter(save_dir, 'stokes')

## Reconstruct Batch

In [ ]:
%%time
for sample in range(len(sample_folders)):

    
    zarr_store = init_zarr_store(save_dir+sample_folders[sample])
    
    for pos in range(len(c1)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        
        zarr_store['C1']['BF'][pos] = BF_stack
        zarr_store['C1']['Retardance'][pos] = ret_stack
        zarr_store['C1']['Orientation'][pos] = ori_stack
        zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))
        

    for pos2 in range(len(c2)):
        
        ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c2[pos2], bg, reconstructor, method = "Tikhonov",
                                                                       reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                       lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
        zarr_store['C2']['BF'][pos2] = BF_stack
        zarr_store['C2']['Retardance'][pos2] = ret_stack
        zarr_store['C2']['Orientation'][pos2] = ori_stack
        zarr_store['C2']['Phase3D'][pos2] = np.transpose(phase3D,(2,0,1))
        

## Reconstruct Single Position

In [ ]:
%%time
sample = 1 #0-5
pos = 0   #0-4

c1, c2, bg = gather_bg_and_positions(data_paths[sample])

zarr_store = init_zarr_store(save_dir+'ReconOrderTest4')

ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(c1[pos], bg, reconstructor, method = "Tikhonov",
                                                               reg_re = 1e-4, reg_im = 1e-4,rho = 1e-5, 
                                                               lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)

# zarr_store['C1']['BF'][pos] = BF_stack
# zarr_store['C1']['Retardance'][pos] = ret_stack
# zarr_store['C1']['Orientation'][pos] = ori_stack
zarr_store['C1']['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))

In [ ]:
## View Retardance
wo.visual.image_stack_viewer_fast(ret_stack, vrange=(0,20))

In [ ]:
## View Phase
wo.visual.image_stack_viewer_fast(np.transpose(phase3D,(2,0,1)))